# 06 BGE-M3 blogger embedding worker

Encode one exact job artifact in the isolated 1024-dimensional BGE-M3 space.

This private `orchestrator_protected` notebook is generated from a reviewed Python template. It receives exact input versions and secrets through Kaggle runtime inputs; no credential is embedded in this notebook or written to its output.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import subprocess
import sys
from pathlib import Path

EXPECTED_SOURCE_SHA256 = 'dafba3b1969f3234365872fa27fcdc5f6e32c863dc655f21c04fe295ec705dc8'
RUNTIME_CONTRACT = 'my-data-hub-blogger-embedding-artifact.v1'
PIN_CONTRACT = {'schema': 'my-data-hub-notebook-execution-pins/v1', 'notebook': '06-bge-m3-blogger-embedding-worker', 'supported_python_series': '3.12', 'kaggle_runtime_image_identity': 'required-immutable-sha256-at-launch', 'input_dataset_versions': 'required-exact-numeric-private-refs-at-launch', 'immutable_assets': ['my_data_hub_wheel_sha256', 'primary_source_sha256'], 'output_contract': 'my-data-hub-blogger-embedding-artifact.v1', 'model': {'id': 'BAAI/bge-m3', 'revision': '5617a9f61b028005a4858fdac845db406aefb181'}, 'privacy': 'private', 'resource_class': 'orchestrator_protected', 'cleanup_retention_policy': {'cleanup_receipt_required': True, 'notebook_resource': 'orchestrator_protected_until_owner_supersedes', 'run_outputs': 'retain_until_terminal_receipt_then_control_policy', 'task_owned_inputs': 'claim_bound_delete_after_terminal_or_expiry'}}
pin_path = Path(os.environ.get('MY_DATA_HUB_EXECUTION_PINS_PATH', ''))
expected_pin_sha = os.environ.get('MY_DATA_HUB_EXECUTION_PINS_SHA256', '')
if not pin_path.is_file() or not re.fullmatch(r'[a-f0-9]{64}', expected_pin_sha):
    raise RuntimeError('hashed execution pins manifest is required')
pin_bytes = pin_path.read_bytes()
if hashlib.sha256(pin_bytes).hexdigest() != expected_pin_sha:
    raise RuntimeError('execution pins manifest hash mismatch')
pins = json.loads(pin_bytes)
required_pin_keys = {
    'schema', 'notebook', 'python_version', 'kaggle_runtime_image_identity',
    'input_dataset_versions', 'immutable_asset_sha256s', 'output_contract',
    'model', 'privacy', 'resource_class', 'cleanup_retention_policy',
}
if not isinstance(pins, dict) or set(pins) != required_pin_keys:
    raise RuntimeError('execution pins manifest keys differ from the exact contract')
if pins['schema'] != PIN_CONTRACT['schema'] or pins['notebook'] != PIN_CONTRACT['notebook']:
    raise RuntimeError('execution pins manifest targets a different notebook contract')
python_version = platform.python_version()
if (pins['python_version'] != python_version or
        not python_version.startswith(PIN_CONTRACT['supported_python_series'] + '.')):
    raise RuntimeError('CPython patch version differs from execution pins')
image_identity = os.environ.get('MY_DATA_HUB_KAGGLE_RUNTIME_IMAGE_IDENTITY', '')
if (pins['kaggle_runtime_image_identity'] != image_identity or
        not re.fullmatch(r'[^@\s]+@sha256:[a-f0-9]{64}', image_identity)):
    raise RuntimeError('immutable Kaggle runtime image identity is required')
dataset_versions = pins['input_dataset_versions']
if (not isinstance(dataset_versions, list) or not dataset_versions or
        any(not isinstance(ref, str) or not re.fullmatch(
            r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+/[1-9][0-9]*', ref
        ) for ref in dataset_versions) or
        len(dataset_versions) != len(set(dataset_versions))):
    raise RuntimeError('exact numeric input Dataset versions are required')
try:
    observed_dataset_versions = json.loads(
        os.environ.get('MY_DATA_HUB_INPUT_DATASET_VERSIONS_JSON', '')
    )
except json.JSONDecodeError as exc:
    raise RuntimeError('observed input Dataset versions are required') from exc
if observed_dataset_versions != dataset_versions:
    raise RuntimeError('attached input Dataset versions differ from execution pins')
if os.environ.get('MY_DATA_HUB_NOTEBOOK_IS_PRIVATE') != 'true':
    raise RuntimeError('operational notebook must be provider-confirmed private')
for key in ('output_contract', 'model', 'privacy', 'resource_class', 'cleanup_retention_policy'):
    if pins[key] != PIN_CONTRACT[key]:
        raise RuntimeError(f'execution pins {key} differs from the generated contract')
wheel = Path(os.environ.get('MY_DATA_HUB_WHEEL_PATH', ''))
if not wheel.is_file() or wheel.suffix != '.whl':
    raise RuntimeError('exact private my-data-hub wheel input is required')
expected_wheel_sha = os.environ.get('MY_DATA_HUB_WHEEL_SHA256', '')
if (len(expected_wheel_sha) != 64 or 
        hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_wheel_sha):
    raise RuntimeError('my-data-hub wheel hash mismatch')
expected_assets = {
    'my_data_hub_wheel_sha256': expected_wheel_sha,
    'primary_source_sha256': EXPECTED_SOURCE_SHA256,
}
if pins['immutable_asset_sha256s'] != expected_assets:
    raise RuntimeError('immutable dependency/source asset hashes differ from execution pins')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--disable-pip-version-check', str(wheel)],
    check=True,
)

In [ ]:
PRIMARY_SOURCE = '"""Primary runtime source for exact-revision BGE-M3 dense-only encoding."""\n\nfrom __future__ import annotations\n\nimport os\nfrom datetime import UTC, datetime\nfrom pathlib import Path\nfrom uuid import UUID\n\nfrom FlagEmbedding import BGEM3FlagModel\nfrom huggingface_hub import snapshot_download\n\nfrom my_data_hub.embeddings.direct_plane import (\n    claim_direct_embedding_jobs, submit_direct_embedding_result,\n)\nfrom my_data_hub.embeddings.models import BGE_M3\nfrom my_data_hub.embeddings.worker import EmbeddingWorker\nfrom my_data_hub.hashing import canonical_json_bytes\n\n\nclass BgeM3Encoder:\n    def __init__(self) -> None:\n        snapshot_path = Path(\n            snapshot_download(repo_id=BGE_M3.model_key, revision=BGE_M3.revision)\n        ).resolve()\n        if snapshot_path.name != BGE_M3.revision:\n            raise RuntimeError(\n                "resolved BGE-M3 snapshot does not match the receipt-bound exact revision"\n            )\n        self.snapshot_revision = snapshot_path.name\n        self.model = BGEM3FlagModel(\n            str(snapshot_path), normalize_embeddings=True, use_fp16=False\n        )\n\n    def encode(self, texts, *, model, max_tokens, pooling, normalize, dense_only):  # type: ignore[no-untyped-def]\n        if model != BGE_M3 or pooling != "model_native_dense" or not normalize or not dense_only:\n            raise ValueError("BGE-M3 runtime contract mismatch")\n        result = self.model.encode(list(texts), batch_size=4, max_length=max_tokens, return_dense=True, return_sparse=False, return_colbert_vecs=False)\n        return result["dense_vecs"].tolist()\n\n\ndef main() -> int:\n    import psycopg\n\n    request_id = UUID(os.environ["MY_DATA_HUB_EMBEDDING_REQUEST_ID"])\n    task_run_id = UUID(os.environ["MY_DATA_HUB_RUN_ID"])\n    input_jobs_sha256 = os.environ["MY_DATA_HUB_EMBEDDING_INPUT_JOBS_SHA256"]\n    # The short-lived URL is injected only through the private per-run status\n    # Dataset and is never written to output or callbacks.\n    with psycopg.connect(os.environ["MY_DATA_HUB_EMBEDDING_DIRECT_DATABASE_URL"]) as connection:\n        jobs = claim_direct_embedding_jobs(\n            connection, request_id=request_id, task_run_id=task_run_id,\n            input_jobs_sha256=input_jobs_sha256,\n        )\n        now = datetime.now(UTC)\n        result = EmbeddingWorker(model=BGE_M3, encoder=BgeM3Encoder()).run(\n            run_id=task_run_id, jobs=jobs, started_at=now, completed_at=datetime.now(UTC)\n        )\n        artifact_sha256 = submit_direct_embedding_result(\n            connection, request_id=request_id, task_run_id=task_run_id,\n            input_jobs_sha256=input_jobs_sha256, manifest=result,\n        )\n    Path("/kaggle/working/embedding-result-metadata.json").write_bytes(\n        canonical_json_bytes({\n            "schema_version": "embedding-direct-result-metadata.v1",\n            "request_id": str(request_id), "task_run_id": str(task_run_id),\n            "input_jobs_sha256": input_jobs_sha256, "artifact_sha256": artifact_sha256,\n        })\n    )\n    return 0\n'
if hashlib.sha256(PRIMARY_SOURCE.encode()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('embedded primary source hash mismatch')
exec(compile(PRIMARY_SOURCE, '<my-data-hub-primary-source>', 'exec'), globals())

In [ ]:
raise SystemExit(globals()['main']())